# Processing all the data for LungCancer

In [12]:
from pathlib import Path
import pandas as pd
import shutil
from tqdm import tqdm
from sklearn.utils import resample

## Configuration

In [13]:
# Paths
BASE_DIR = Path('../')
LungCancerDetection_DIR = BASE_DIR / 'data' / 'LungCancerDetection' / 'processed'
# Utiliser le lien vers data mais moi c'est labah 
LungCancerDetection_Raw_data = Path("C:/Users/DELL/.cache/kagglehub/datasets/ashery/chexpert/versions/1")

## Load Dataset

### Preprocessing Dataset

In [14]:
#Loading the csv file
df = pd.read_csv(LungCancerDetection_Raw_data/"train.csv")

df.head()

,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0
1,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87,Frontal,AP,NaN,NaN,-1.0,1.0,NaN,-1.0,-1.0,NaN,-1.0,NaN,-1.0,NaN,1.0,NaN
2,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Frontal,AP,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN
3,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Lateral,NaN,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN
4,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,Frontal,AP,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 223414 entries, 0 to 223413
Data columns (total 19 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Path                        223414 non-null  object 
 1   Sex                         223414 non-null  object 
 2   Age                         223414 non-null  int64  
 3   Frontal/Lateral             223414 non-null  object 
 4   AP/PA                       191027 non-null  object 
 5   No Finding                  22381 non-null   float64
 6   Enlarged Cardiomediastinum  44839 non-null   float64
 7   Cardiomegaly                46203 non-null   float64
 8   Lung Opacity                117778 non-null  float64
 9   Lung Lesion                 11944 non-null   float64
 10  Edema                       85956 non-null   float64
 11  Consolidation               70622 non-null   float64
 12  Pneumonia                   27608 non-null   float64
 13  Atelectasis   

In [16]:
print(df['Lung Lesion'].value_counts(dropna=False))
print("\n" + "="*50)
print(df['Lung Lesion'].describe())

Lung Lesion
 NaN    211470
 1.0      9186
-1.0      1488
 0.0      1270
Name: count, dtype: int64

count    11944.000000
mean         0.644508
std          0.691607
min         -1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: Lung Lesion, dtype: float64


As explianed in the paper, we will only use the Lung Lesion Column which has 3 differents values : 

-1 (no lesion)
1 (lesion)
0 (not sure)

We wanted to put -1 in begnin and 1 for malignant.

In [17]:
# Filtrer pour garder uniquement les images avec lung lesion confirmée ou absence confirmée
df2 = df[df['Lung Lesion'].notna()].copy()

print(f"Total images avec lung lesion label: {len(df2)}")
print(f"Malignant (Lung Lesion=1): {(df2['Lung Lesion'] == 1).sum()}")
print(f"Benign (Lung Lesion=0 ou -1): {(df2['Lung Lesion'] == 0).sum() + (df['Lung Lesion'] == -1).sum()}")

Total images avec lung lesion label: 11944
Malignant (Lung Lesion=1): 9186
Benign (Lung Lesion=0 ou -1): 2758


There is supposed to be the same number of malignant and Benign, so we are going to use sub balancing. however there would be not enought data thus we also are going to use the No finding column to add more benign data.

In [18]:
benign_mask = (df['Lung Lesion'] == 0.0) | (df['No Finding'] == 1.0)
malignant_mask = (df['Lung Lesion'] == 1.0)

df['lung_lesion'] = malignant_mask.astype(int)  # 1 = malignant confirmé, 0 = benign

# Garder seulement les images avec un label clair
df = df[benign_mask | malignant_mask].copy()

print(f"Total images: {len(df)}")
print(f"Benign: {(df['lung_lesion'] == 0).sum()}")
print(f"Malignant: {(df['lung_lesion'] == 1).sum()}")

# Subsampling pour équilibrer

benign = df[df['lung_lesion'] == 0]
malignant = df[df['lung_lesion'] == 1]
min_size = min(len(benign), len(malignant))

benign_balanced = resample(benign, n_samples=min_size, random_state=42)
malignant_balanced = resample(malignant, n_samples=min_size, random_state=42)

df_balanced = pd.concat([benign_balanced, malignant_balanced]).reset_index(drop=True)

print(f"\nAprès équilibrage:")
print(f"Benign: {(df_balanced['lung_lesion'] == 0).sum()}")
print(f"Malignant: {(df_balanced['lung_lesion'] == 1).sum()}")
print(f"Total: {len(df_balanced)}")

Total images: 32460
Benign: 23274
Malignant: 9186

Après équilibrage:
Benign: 9186
Malignant: 9186
Total: 18372


In [19]:
df_balanced = df_balanced[['Path', 'lung_lesion']].reset_index(drop=True)

# Enlever le préfixe CheXpert-v1.0-small/ du début des chemins
df_balanced['Path'] = df_balanced['Path'].str.replace('CheXpert-v1.0-small/', '', regex=False)

df_balanced.head()

,Path,lung_lesion
0,train/patient31583/study1/view1_frontal.jpg,0
1,train/patient01699/study2/view1_frontal.jpg,0
2,train/patient10847/study1/view2_lateral.jpg,0
3,train/patient55000/study1/view1_frontal.jpg,0
4,train/patient24237/study3/view1_frontal.jpg,0


## Split dataset

In [20]:
from sklearn.model_selection import train_test_split

# Split 70% train, 15% val, 15% test avec stratification
train_df, temp_df = train_test_split(df_balanced, test_size=0.3, stratify=df_balanced['lung_lesion'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['lung_lesion'], random_state=42)

print(f"Train: {len(train_df)} ({(train_df['lung_lesion']==1).sum()} mal, {(train_df['lung_lesion']==0).sum()} ben)")
print(f"Val: {len(val_df)} ({(val_df['lung_lesion']==1).sum()} mal, {(val_df['lung_lesion']==0).sum()} ben)")
print(f"Test: {len(test_df)} ({(test_df['lung_lesion']==1).sum()} mal, {(test_df['lung_lesion']==0).sum()} ben)")

Train: 12860 (6430 mal, 6430 ben)
Val: 2756 (1378 mal, 1378 ben)
Test: 2756 (1378 mal, 1378 ben)


## Copier les images dans les répertoires train, validation et test

In [21]:
# Créer les répertoires de destination
for split in ['training', 'validation', 'testing']:
    for label in ['benign', 'malignant']:
        (LungCancerDetection_DIR / split / label).mkdir(parents=True, exist_ok=True)

print("Répertoires créés:")
print(f"- {LungCancerDetection_DIR / 'training' / 'benign'}")
print(f"- {LungCancerDetection_DIR / 'training' / 'malignant'}")
print(f"- {LungCancerDetection_DIR / 'validation' / 'benign'}")
print(f"- {LungCancerDetection_DIR / 'validation' / 'malignant'}")
print(f"- {LungCancerDetection_DIR / 'testing' / 'benign'}")
print(f"- {LungCancerDetection_DIR / 'testing' / 'malignant'}")

Répertoires créés:
- ..\data\LungCancerDetection\processed\training\benign
- ..\data\LungCancerDetection\processed\training\malignant
- ..\data\LungCancerDetection\processed\validation\benign
- ..\data\LungCancerDetection\processed\validation\malignant
- ..\data\LungCancerDetection\processed\testing\benign
- ..\data\LungCancerDetection\processed\testing\malignant


In [22]:
def copy_images(df, split_name, source_dir, dest_dir):
    """
    Copie les images d'un dataframe vers le répertoire de destination
    
    Args:
        df: DataFrame contenant les colonnes 'Path' et 'lung_lesion'
        split_name: nom du split ('train', 'validation', 'test')
        source_dir: répertoire source contenant les images originales
        dest_dir: répertoire de destination
    """
    copied = 0
    skipped = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Copie {split_name}"):
        # Déterminer la classe
        label = 'malignant' if row['lung_lesion'] == 1 else 'benign'
        
        # Construire les chemins
        source_path = source_dir / row['Path']
        
        # Créer un nom de fichier unique pour éviter les collisions
        filename = f"{idx}_{Path(row['Path']).name}"
        dest_path = dest_dir / split_name / label / filename
        
        # Copier le fichier si la source existe
        if source_path.exists():
            shutil.copy2(source_path, dest_path)
            copied += 1
        else:
            skipped += 1
            if skipped <= 5:  # Afficher seulement les 5 premiers fichiers manquants
                print(f"  Fichier introuvable: {source_path}")
    
    print(f"{split_name}: {copied} images copiées, {skipped} fichiers manquants")
    return copied, skipped

In [23]:
# Copier les images pour chaque split
print("Début de la copie des images...\n")

# Train
train_copied, train_skipped = copy_images(train_df, 'training', LungCancerDetection_Raw_data, LungCancerDetection_DIR)

# Validation
val_copied, val_skipped = copy_images(val_df, 'validation', LungCancerDetection_Raw_data, LungCancerDetection_DIR)

# Test
test_copied, test_skipped = copy_images(test_df, 'testing', LungCancerDetection_Raw_data, LungCancerDetection_DIR)

print("\n" + "="*60)
print("Résumé de la copie:")
print(f"Train: {train_copied} copiées / {len(train_df)} total")
print(f"Validation: {val_copied} copiées / {len(val_df)} total")
print(f"Test: {test_copied} copiées / {len(test_df)} total")
print(f"\nTotal copié: {train_copied + val_copied + test_copied}")
print(f"Total manquant: {train_skipped + val_skipped + test_skipped}")

Début de la copie des images...



Copie training: 100%|██████████| 12860/12860 [00:23<00:00, 545.04it/s]


training: 12860 images copiées, 0 fichiers manquants


Copie validation: 100%|██████████| 2756/2756 [00:06<00:00, 423.62it/s]


validation: 2756 images copiées, 0 fichiers manquants


Copie testing: 100%|██████████| 2756/2756 [00:07<00:00, 373.49it/s]

testing: 2756 images copiées, 0 fichiers manquants

Résumé de la copie:
Train: 12860 copiées / 12860 total
Validation: 2756 copiées / 2756 total
Test: 2756 copiées / 2756 total

Total copié: 18372
Total manquant: 0
